In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col,
    lit,
    from_json,
    to_timestamp,
    unix_timestamp,
    current_timestamp,
    concat_ws,
    when,
    abs
)
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType
)

# ============================================================
# SPARK SESSION
# ============================================================

spark = (
    SparkSession.builder
    .appName("RealTimeTaxiDataQuality")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")


# ============================================================
# KAFKA CONFIGURATION
# ============================================================

KAFKA_SERVER = "kafka:9092"
KAFKA_TOPIC = "taxi-data"


# ============================================================
# READ KAFKA STREAM
# ============================================================

raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_SERVER)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", "earliest")
    .option("failOnDataLoss", "false")
    .load()
)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col,
    lit,
    from_json,
    to_timestamp,
    unix_timestamp,
    current_timestamp,
    concat_ws,
    when,
    abs
)
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType
)

# ============================================================
# SPARK SESSION
# ============================================================

spark = (
    SparkSession.builder
    .appName("RealTimeTaxiDataQuality")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")


# ============================================================
# KAFKA CONFIGURATION
# ============================================================

KAFKA_SERVER = "kafka:9092"
KAFKA_TOPIC = "taxi-data"


# ============================================================
# READ KAFKA STREAM
# ============================================================

raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_SERVER)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", "earliest")
    .option("failOnDataLoss", "false")
    .load()
)

In [ ]:
# ============================================================
# PARSE JSON
# ============================================================

events = raw_stream.select(

    from_json(
        col("value").cast("string"),
        taxi_schema
    ).alias("data"),

    col("partition"),

    col("topic"),

    col("offset"),

    col("timestamp").alias("kafka_timestamp")
)


# Flatten JSON struct

events = events.select(
    "data.*",
    "partition",
    "topic",
    "offset",
    "kafka_timestamp"
)

In [ ]:
# ============================================================
# PARSE JSON
# ============================================================

events = raw_stream.select(

    from_json(
        col("value").cast("string"),
        taxi_schema
    ).alias("data"),

    col("partition"),

    col("topic"),

    col("offset"),

    col("timestamp").alias("kafka_timestamp")
)


# Flatten JSON struct

events = events.select(
    "data.*",
    "partition",
    "topic",
    "offset",
    "kafka_timestamp"
)

In [ ]:
# ============================================================
# TIMESTAMP CONVERSION
# ============================================================

events = (
    events
    .withColumn(
        "ptime",
        to_timestamp("ptime")
    )
    .withColumn(
        "dtime",
        to_timestamp("dtime")
    )
)

In [ ]:
# ============================================================
# TAXI ZONE LOOKUP
# ============================================================

zone_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("../data/taxi_zone_lookup.csv")
)

zone_df.printSchema()

In [ ]:
# ============================================================
# PICKUP LOOKUP
# ============================================================

pickup_zone = zone_df.select(

    col("LocationID")
        .cast("int")
        .alias("PULocationID"),

    col("Borough")
        .alias("PU_Borough"),

    col("Zone")
        .alias("PU_Zone"),

    col("service_zone")
        .alias("PU_service_zone")
)

In [ ]:
# ============================================================
# DROPOFF LOOKUP
# ============================================================

dropoff_zone = zone_df.select(

    col("LocationID")
        .cast("int")
        .alias("DOLocationID"),

    col("Borough")
        .alias("DO_Borough"),

    col("Zone")
        .alias("DO_Zone"),

    col("service_zone")
        .alias("DO_service_zone")
)

In [ ]:
# ============================================================
# DROPOFF LOCATION ENRICHMENT
# ============================================================

events = events.join(
    dropoff_zone,
    on="DOLocationID",
    how="left"
)
# ============================================================
# PICKUP LOCATION ENRICHMENT
# ============================================================

events = events.join(
    pickup_zone,
    on="PULocationID",
    how="left"
)



In [ ]:
# ============================================================
# CALCULATED DURATION
# ============================================================

events = events.withColumn(
    "calculated_duration",
    unix_timestamp("dtime") -
    unix_timestamp("ptime")
)

In [ ]:
# ============================================================
# REQUIRED FIELD VALIDATION
# ============================================================

events = events.withColumn(
    "invalid_required_fields",

    (
        col("event_id").isNull() |

        col("VendorID").isNull() |

        col("passenger_count").isNull() |

        col("trip_distance").isNull() |

        col("PULocationID").isNull() |

        col("DOLocationID").isNull() |

        col("payment_type").isNull() |

        col("fare_amount").isNull() |

        col("total_amount").isNull() |

        col("total_time_in_sec").isNull() |

        col("ptime").isNull() |

        col("dtime").isNull()
    )
)

In [ ]:
# ============================================================
# VENDOR VALIDATION
# ============================================================

events = events.withColumn(
    "invalid_vendor",

    (
        col("VendorID").isNull() |
        ~col("VendorID").isin(1, 2)
    )
)

# ============================================================
# PASSENGER COUNT VALIDATION
# ============================================================

events = events.withColumn(
    "invalid_passenger_count",

    (
        col("passenger_count").isNull() |
        (col("passenger_count") <= 0)
    )
)
# ============================================================
# DISTANCE VALIDATION
# ============================================================

events = events.withColumn(
    "invalid_distance",

    (
        col("trip_distance").isNull() |
        (col("trip_distance") < 0)
    )
)

# ============================================================
# RATE CODE VALIDATION
# ============================================================

events = events.withColumn(
    "invalid_ratecode",

    (
        col("RatecodeID").isNull() |
        ~col("RatecodeID").isin(
            1, 2, 3, 4, 5, 6, 99
        )
    )
)
# ============================================================
# PICKUP LOCATION VALIDATION
# ============================================================

events = events.withColumn(
    "invalid_pickup_location",

    col("PU_Zone").isNull()
)

# ============================================================
# DROPOFF LOCATION VALIDATION
# ============================================================

events = events.withColumn(
    "invalid_dropoff_location",

    col("DO_Zone").isNull()
)


# ============================================================
# PAYMENT TYPE VALIDATION
# ============================================================

events = events.withColumn(
    "invalid_payment_type",

    (
        col("payment_type").isNull() |
        ~col("payment_type").isin(
            1, 2, 3, 4, 5, 6
        )
    )
)

# ============================================================
# FARE VALIDATION
# ============================================================

events = events.withColumn(
    "invalid_fare",

    (
        col("fare_amount").isNull() |
        (col("fare_amount") < 0)
    )
)

# ============================================================
# EXTRA VALIDATION
# ============================================================

events = events.withColumn(
    "invalid_extra",

    (
        col("extra").isNull() |
        (col("extra") < 0)
    )
)

# ============================================================
# MTA TAX VALIDATION
# ============================================================

events = events.withColumn(
    "invalid_mta_tax",

    (
        col("mta_tax").isNull() |
        (col("mta_tax") < 0)
    )
)

# ============================================================
# TIP VALIDATION
# ============================================================

events = events.withColumn(
    "invalid_tip",

    (
        col("tip_amount").isNull() |
        (col("tip_amount") < 0)
    )
)

# ============================================================
# TIP VALIDATION
# ============================================================

events = events.withColumn(
    "invalid_tip",

    (
        col("tip_amount").isNull() |
        (col("tip_amount") < 0)
    )
)

# ============================================================
# IMPROVEMENT SURCHARGE VALIDATION
# ============================================================

events = events.withColumn(
    "invalid_improvement_surcharge",

    (
        col("improvement_surcharge").isNull() |
        (col("improvement_surcharge") < 0)
    )
)

# ============================================================
# IMPROVEMENT SURCHARGE VALIDATION
# ============================================================

events = events.withColumn(
    "invalid_improvement_surcharge",

    (
        col("improvement_surcharge").isNull() |
        (col("improvement_surcharge") < 0)
    )
)

# ============================================================
# DURATION VALIDATION
# ============================================================

events = events.withColumn(
    "invalid_duration",

    (
        col("total_time_in_sec").isNull() |
        (col("total_time_in_sec") <= 0)
    )
)

# ============================================================
# TIME ORDER VALIDATION
# ============================================================

events = events.withColumn(
    "invalid_time_order",

    (
        col("ptime").isNull() |
        col("dtime").isNull() |
        (col("dtime") < col("ptime"))
    )
)

# ============================================================
# DURATION CONSISTENCY
# ============================================================

events = events.withColumn(
    "invalid_duration_consistency",

    (
        col("calculated_duration").isNull() |

        (
            abs(
                col("total_time_in_sec") -
                col("calculated_duration")
            ) > 1
        )
    )
)

# ============================================================
# FUTURE TIMESTAMP VALIDATION
# ============================================================

events = events.withColumn(
    "invalid_future_timestamp",

    (
        (col("ptime") > current_timestamp()) |
        (col("dtime") > current_timestamp())
    )
)

# ============================================================
# SUSPICIOUS DATA
# ============================================================

events = (
    events

    .withColumn(
        "suspicious_zero_distance",
        col("trip_distance") == 0
    )

    .withColumn(
        "suspicious_zero_fare",
        col("total_amount") == 0
    )

    .withColumn(
        "suspicious_high_passenger_count",
        col("passenger_count") > 10
    )

    .withColumn(
        "suspicious_high_fare",
        col("total_amount") > 500
    )
)

In [ ]:
# ============================================================
# VALIDATION ERROR MESSAGE
# ============================================================

events = events.withColumn(
    "validation_errors",

    concat_ws(

        ", ",

        when(
            col("invalid_required_fields"),
            lit("missing_required_field")
        ),

        when(
            col("invalid_vendor"),
            lit("invalid_vendor_id")
        ),

        when(
            col("invalid_passenger_count"),
            lit("invalid_passenger_count")
        ),

        when(
            col("invalid_distance"),
            lit("negative_distance")
        ),

        when(
            col("invalid_ratecode"),
            lit("invalid_ratecode")
        ),

        when(
            col("invalid_pickup_location"),
            lit("invalid_pickup_location")
        ),

        when(
            col("invalid_dropoff_location"),
            lit("invalid_dropoff_location")
        ),

        when(
            col("invalid_payment_type"),
            lit("invalid_payment_type")
        ),

        when(
            col("invalid_fare"),
            lit("negative_fare")
        ),

        when(
            col("invalid_extra"),
            lit("invalid_extra")
        ),

        when(
            col("invalid_mta_tax"),
            lit("invalid_mta_tax")
        ),

        when(
            col("invalid_tip"),
            lit("invalid_tip")
        ),

        when(
            col("invalid_tolls"),
            lit("invalid_tolls")
        ),

        when(
            col("invalid_improvement_surcharge"),
            lit("invalid_improvement_surcharge")
        ),

        when(
            col("invalid_total_amount"),
            lit("invalid_total_amount")
        ),

        when(
            col("invalid_duration"),
            lit("invalid_duration")
        ),

        when(
            col("invalid_time_order"),
            lit("dropoff_before_pickup")
        ),

        when(
            col("invalid_duration_consistency"),
            lit("duration_mismatch")
        ),

        when(
            col("invalid_future_timestamp"),
            lit("future_timestamp")
        )
    )
)

In [ ]:
# ============================================================
# OVERALL VALIDATION
# ============================================================

events = events.withColumn(
    "is_valid",

    ~(
        col("invalid_required_fields") |

        col("invalid_vendor") |

        col("invalid_passenger_count") |

        col("invalid_distance") |

        col("invalid_ratecode") |

        col("invalid_pickup_location") |

        col("invalid_dropoff_location") |

        col("invalid_payment_type") |

        col("invalid_fare") |

        col("invalid_extra") |

        col("invalid_mta_tax") |

        col("invalid_tip") |

        col("invalid_tolls") |

        col("invalid_improvement_surcharge") |

        col("invalid_total_amount") |

        col("invalid_duration") |

        col("invalid_time_order") |

        col("invalid_duration_consistency") |

        col("invalid_future_timestamp")
    )
)

In [ ]:
# ============================================================
# DUPLICATE EVENT DETECTION
# ============================================================

duplicate_ids = (
    events
    .groupBy("event_id")
    .count()
    .filter(col("count") > 1)
    .select("event_id")
    .withColumn(
        "is_duplicate",
        lit(True)
    )
)

In [ ]:
# ============================================================
# MARK DUPLICATES
# ============================================================

events = (
    events
    .join(
        duplicate_ids,
        on="event_id",
        how="left"
    )
    .withColumn(
        "is_duplicate",
        F.coalesce(
            col("is_duplicate"),
            lit(False)
        )
    )
)

In [ ]:
# ============================================================
# BRONZE
# ============================================================

bronze_df = events

In [ ]:
# ============================================================
# SILVER
# ============================================================

silver_df = bronze_df.filter(
    (col("is_valid") == True) &
    (col("is_duplicate") == False)
)

In [ ]:
# ============================================================
# QUARANTINE
# ============================================================

quarantine_df = bronze_df.filter(
    col("is_valid") == False
)

# ============================================================
# DUPLICATES
# ============================================================

duplicate_df = bronze_df.filter(
    col("is_duplicate") == True
)
bronze_df.select(
    "event_id",
    "VendorID",
    "passenger_count",
    "trip_distance",
    "PULocationID",
    "PU_Zone",
    "DOLocationID",
    "DO_Zone",
    "payment_type",
    "total_amount",
    "ptime",
    "dtime",
    "calculated_duration",
    "validation_errors",
    "is_valid",
    "is_duplicate"
).show(
    20,
    truncate=False
)



In [ ]:
bronze_df.groupBy(
    "is_valid"
).count().show()

In [ ]:
bronze_df.groupBy(
    "validation_errors"
).count().orderBy(
    F.desc("count")
).show(
    truncate=False
)

In [ ]:
silver_df.count()

In [ ]:
quarantine_df.count()